In [1]:
import cv2
import os
import matplotlib.pyplot as plt
import numpy as np
from google.colab.patches import cv2_imshow
import tensorflow as tf

dataset path

In [11]:


# New processed dataset
dataset_path = "/content/drive/MyDrive/Sindhi handwritten Numeral recognition system/dataset"

new_processed_dataset_path = "/content/drive/MyDrive/Sindhi handwritten Numeral recognition system/New_processed_dataset"



create preprocessing function

In [12]:
def preprocess_image(image_path):

    # Read image
    image = cv2.imread(image_path)

    if image is None:
        return None

    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Gaussian Blur
    blurred = cv2.GaussianBlur(gray, (5,5), 0)

    # Adaptive Threshold
    threshold = cv2.adaptiveThreshold(
        blurred,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        11,
        2
    )

    # Invert image
    inverted = cv2.bitwise_not(threshold)

    # Kernel
    kernel = np.ones((3,3), np.uint8)

    # Opening
    opened = cv2.morphologyEx(
        inverted,
        cv2.MORPH_OPEN,
        kernel
    )

    # Closing
    closed = cv2.morphologyEx(
        opened,
        cv2.MORPH_CLOSE,
        kernel
    )

    # return closed

        # Find contours
    contours, _ = cv2.findContours(
        closed,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # No contour found
    if len(contours) == 0:
        return None

    # Largest contour
    largest_contour = max(contours, key=cv2.contourArea)

    # return largest_contour

    x, y, w, h = cv2.boundingRect(largest_contour)

    padding = 10

    x = max(0, x - padding)
    y = max(0, y - padding)

    w = min(closed.shape[1] - x, w + 2 * padding)
    h = min(closed.shape[0] - y, h + 2 * padding)

    cropped_digit = closed[y:y+h, x:x+w]

    resized_digit = cv2.resize(cropped_digit, (64, 64))

    return resized_digit

process entire dataset

In [13]:
for folder in os.listdir(dataset_path):

    source_folder = os.path.join(dataset_path, folder)

    # Skip if not a folder
    if not os.path.isdir(source_folder):
        continue

    destination_folder = os.path.join(new_processed_dataset_path, folder)

    os.makedirs(destination_folder, exist_ok=True)

    print(f"Processing Folder: {folder}")

    for image_name in os.listdir(source_folder):

        image_path = os.path.join(source_folder, image_name)

        # Preprocess image
        processed_image = preprocess_image(image_path)

        # Skip invalid images
        if processed_image is None:
            print(f"Skipped: {image_name}")
            continue

        # Save processed image
        save_path = os.path.join(destination_folder, image_name)

        cv2.imwrite(save_path, processed_image)

print("\n✅ Dataset preprocessing completed successfully!")

Processing Folder: 5
Processing Folder: 9
Processing Folder: 1
Processing Folder: 6
Processing Folder: 7
Processing Folder: 8
Processing Folder: 4
Processing Folder: 3
Processing Folder: 2
Processing Folder: 0

✅ Dataset preprocessing completed successfully!
